# 疾患関連遺伝子のランキング

疾患名と遺伝子リストを**変数**として入れると、候補遺伝子を関連度順に並べ替えるノートブック。

## このノートブックが守っていること

`GPR52` を答えてほしいのに `GPR56` が返る、という失敗があります。これは知識不足ではなく
**文字の生成のしかた**の問題です。どのトークナイザも `GPR52` を 1 つのかたまりでは持たず、
`G` / `PR` / `5` / `2` のように分割します。`GPR` まで出力した時点で、次に来る文字は
学習データでの出現回数に強く引っ張られます。論文に多く出てくる `GPR56` が勝ちます。
プロンプトに候補リストを書いても、それは「参考情報」であって「制約」ではありません。

対策はプロンプトの書き方ではなく、**構成そのもの**です。

| 段階 | 遺伝子リストが何になるか | 記号がどこにあるか |
|---|---|---|
| 1 | 遺伝子ごとに独立した点数付け（PMI） | 入力側（採点される） |
| 2 | 順序を入れ替えた A〜E の選択肢 | 選択肢（出力は 1 文字だけ） |

どちらもモデルが遺伝子記号を**書く**ことはありません。だから `GPR52` と `GPR56` は
別々の点数を持ち、混ざりようがありません。

## 実行の順番

1. 〜 3. はモデルなしで動きます（プロンプトの確認まで）。
4. 以降は `MODEL` を設定して GPU のある環境で実行してください。

## 0. セットアップ

In [ ]:
import os, sys, json, importlib

# scripts/ を import できるようにする（このノートブックは notebooks/ にある想定）
SKILL = os.path.abspath(os.path.join("..", ".claude", "skills", "gene-disease-ranking"))
if not os.path.isdir(SKILL):                      # ノートブックだけ別の場所に置いた場合
    SKILL = os.path.abspath(os.environ.get("GDR_SKILL_DIR", "."))
sys.path.insert(0, os.path.join(SKILL, "scripts"))

EXAMPLES = os.path.join(SKILL, "examples")
print("skill dir:", SKILL)

def have(mod):
    return importlib.util.find_spec(mod) is not None

DEPS = {m: have(m) for m in
        ("numpy", "torch", "transformers", "llama_cpp", "pandas")}
for m, ok in DEPS.items():
    print(f"  {m:<14} {'あり' if ok else 'なし'}")

if not DEPS["numpy"]:
    print("\nnumpy が必要です:  pip install numpy")
print()
print("バックエンド別に必要なもの：")
print("  ollama       追加パッケージ不要（HTTP のみ）。Ollama 本体が動いていること")
print("  llamacpp     pip install llama-cpp-python")
print("  transformers pip install torch transformers")
print("プロンプトの確認（3 章まで）は、どれも無しで動きます。")

## 1. 入力（編集するのはここだけ）

`DISEASE` に疾患名、`GENES` に候補遺伝子のリストを入れます。どちらもただの Python 変数です。

`GENES` はわざと**似た記号を混ぜて**あります（GPR52/55/56/35、HBB/HBA1/HBA2、
SLC6A4/SLC6A3）。記号を生成してしまうパイプラインなら、ここで目に見えて失敗します。

In [ ]:
# --- 疾患名（1 つ）---
DISEASE = "Cystic fibrosis"

# --- 候補遺伝子（リスト）---
GENES = [
    "GPR52", "GPR55", "GPR56", "GPR35",
    "HTT", "CFTR",
    "HBB", "HBA1", "HBA2",
    "APOE", "APOC1",
    "TP53",
    "SLC6A4", "SLC6A3",
]

# --- どこで動かすか ---
#   "ollama"       ローカルの Ollama。段階 2 のみ（理由は 3.5 章）
#   "llamacpp"     Ollama が落としてきた GGUF をそのまま使う。段階 1 も動く
#   "transformers" 量子化なしの重み。基準となる実装
BACKEND = "ollama"

# None なら自動探索（localhost → host.docker.internal → 172.17.0.1 の順）。
# Docker の Ollama をホストから使う場合、ポートが公開されていれば
# http://localhost:11434 で届きます。
OLLAMA_HOST = None

# --- 使うモデル。None のままだとモデルを使う段階は飛ばされます ---
MODEL = None
# ollama / llamacpp の場合は `ollama list` に出てくる名前:
#     MODEL = "gemma3:27b"
#     MODEL = "meditron:70b"
# transformers の場合は Hugging Face の ID:
#     MODEL = "EPFLiGHT/Gemma-3-27B-MeditronFO"   # 医療特化 27B、Gemma 条項
#     MODEL = "google/gemma-3-27b-it"             # 素のモデル（対照として必ず測る）
#     MODEL = "EPFLiGHT/Apertus-70B-MeditronFO"   # Apache 2.0、ライセンスは一番緩い

# --- 詰めの設定（最初はそのままで可）---
TOP_K            = None   # 段階 2 に送る件数。None なら候補数から自動決定
GROUP_SIZE       = 4      # 1 問あたりの遺伝子数（+ 逃げ道 1 つで A〜E）
ROTATIONS        = 4      # 選択肢の順序を何通り試すか
MARGIN_THRESHOLD = 0.5    # 1 位と 2 位の差がこれ未満なら「判定しない」

_NEEDS = {"transformers": ("torch", "transformers"),
          "llamacpp": ("llama_cpp",),
          "ollama": ()}                  # HTTP だけ。到達性は 3.5 章で確認
HAVE_MODEL = bool(MODEL) and all(DEPS[m] for m in _NEEDS[BACKEND])

print(f"疾患         : {DISEASE}")
print(f"候補         : {len(GENES)} 個")
print(f"バックエンド : {BACKEND}")
print(f"モデル       : {MODEL or '（未設定 → 3 章まで実行されます）'}")
if MODEL and not HAVE_MODEL:
    print(f"\n{BACKEND} には {', '.join(_NEEDS[BACKEND])} が必要です。")

## 2. リストの整理と、方法の自動選択

候補数によってやり方が変わります。ここは自動です。

| 候補数 | 方法 | 理由 |
|---|---|---|
| 2〜10 | 段階 2 のみ | この規模なら PMI を足す意味が薄い |
| 11〜100 | PMI → 上位 10 件を段階 2 | |
| 100 超 | PMI → 上位 20 件を段階 2 | 長いリストを 1 つのプロンプトに入れても、真ん中はモデルがほぼ見ていない |

In [ ]:
from prompts import clean_genes, plan_stages

genes, norm = clean_genes(GENES)
plan = plan_stages(len(genes), TOP_K)

print(f"入力 {norm['n_input']} 個 → 整理後 {norm['n_output']} 個")
if norm["duplicates"]:
    print("  重複を除去 :", norm["duplicates"])
print()
print("方法 :", " → ".join(plan["stages"]))
print("理由 :", plan["reason"])
print("段階 2 に送る件数 :", plan["shortlist"])

### （任意）HGNC で記号を正規化する

別名を正式名称にそろえ、遺伝子でない文字列を落とします。`HTT` と `IT15` は同じ遺伝子ですが、
そのままだと別々に点数が付いて信号が分散します。

`hgnc_complete_set.txt` は [genenames.org](https://www.genenames.org/download/statistics-and-files/)
から取得します。**手元にすでにデータベースがあるなら、そちらを先に確認してください。**
ダウンロードした場合は取得日を記録しておくこと（古いと後の比較が歪みます）。

なお、この処理で消えるのは「存在しない遺伝子」だけです。`GPR56` は正式名称として実在するので、
**取り違えはこれでは直りません**。それを直すのは記号を生成させない構成のほうです。

In [ ]:
HGNC_PATH = None    # 例: "/path/to/hgnc_complete_set.txt"

if HGNC_PATH and os.path.exists(HGNC_PATH):
    from normalize_genes import load_hgnc
    approved, alias_to = load_hgnc(HGNC_PATH)
    genes, norm = clean_genes(GENES, approved, alias_to, keep_unknown=False)
    plan = plan_stages(len(genes), TOP_K)
    print(f"承認記号 {len(approved)} 件 / 別名 {len(alias_to)} 件を読み込み")
    print(f"整理後 {len(genes)} 個")
    for m in norm["mapped"]:
        print(f"  別名を変換 : {m['input']} → {m['approved']}")
    if norm["unknown"]:
        print("  遺伝子でないため除去 :", norm["unknown"])
else:
    print("HGNC ファイル未指定のため、この段階は飛ばしました。")
    print("記号のつづりは検証されていません。")

## 3. 実際に送られるプロンプトを見る（モデル不要）

ここが一番大事な確認です。**遺伝子記号がモデルの出力側に置かれていないこと**を目で見ます。

In [ ]:
from rank import preview

p = preview(DISEASE, genes, TOP_K, GROUP_SIZE, ROTATIONS)

if p["stage1"]:
    print("=== 段階 1：遺伝子 1 個につき 1 回、独立に採点 ===")
    print(f"  前置き   : {p['stage1']['conditional_prefix']!r}")
    print(f"  採点対象 : {p['genes'][0]!r}  ← 生成させるのではなく、この文字列の確率を測る")
    print(f"  引く項   : {p['stage1']['neutral_prefix']!r} + 同じ対象")
    print()
    print("  引き算の意味：「疾患名を出したことで、この遺伝子の確率がどれだけ上がったか」。")
    print("  ここを引かないと、どの疾患でも TP53・EGFR・TNF が上位に来ます。")
    print(f"  対象は {len(p['genes'])} 個すべて別々に測るので、GPR52 と GPR56 は混ざりません。")
else:
    print("=== 段階 1 は省略（候補が少ないため）===")

In [ ]:
rounds = p["stage2_rounds"]
print(f"=== 段階 2：{len(rounds)} 問。答えは A〜E の 1 文字だけ ===\n")

for rnd in rounds[:2]:
    print(f"--- グループ {rnd['group']} / 順序 {rnd['rotation']} ---")
    print(rnd["prompt"])
    print()

print(f"（残り {max(0, len(rounds) - 2)} 問は省略）")
print()
print("確認すべき点：")
print("  ・プロンプトが 'Answer:' で終わっている → モデルは 1 文字を出すだけ")
print("  ・遺伝子記号は選択肢の側にしかない     → 記号を書かせていない")
print("  ・'None of the above' が必ずある       → 逃げ道。無いと当てずっぽうを強制する")
print("  ・同じ遺伝子が違う位置で何度も出る     → 位置による偏りを打ち消す")

## 3.5 バックエンドの確認 ─ Ollama を使う場合はここが分かれ道

このパイプラインはモデルに 2 つの違うことを頼みます。難しさが全く違います。

| | 何を頼むか | どこで使うか |
|---|---|---|
| A | **こちらが渡した文字列**の確率を教えて | 段階 1（PMI） |
| B | 次の 1 文字の確率を教えて（A〜E） | 段階 2 |

B は「普通の生成＋確率の表示」です。A は生成ではなく**採点**で、これができない実行環境があります。

**Ollama は A ができません。** Ollama の `logprobs` / `top_logprobs`（v0.12.11 以降）は
**モデルが生成したトークン**の確率です。OpenAI の `echo` に相当する機能も、採点用の
エンドポイントもないため、こちらが渡した `GPR52` という文字列の確率を読み出せません。

| バックエンド | 段階 1 | 段階 2 | 備考 |
|---|---|---|---|
| `transformers` | ○ | ○ | 基準となる実装 |
| `llamacpp` | ○ | ○ | GGUF。**Ollama が持っている重みをそのまま読める** |
| `ollama` | **×** | ○ | 渡した記号を採点できない |

**推奨：`llamacpp`。** Ollama は GGUF ファイルを `~/.ollama/models/blobs/` に置いています。
llama.cpp から同じファイルを指せば、**再ダウンロードなしで**段階 1 も動きます。

```bash
pip install llama-cpp-python
```

`ollama` のまま進めることもできます。その場合は段階 2 だけを全候補に対して回します。
遺伝子記号を生成しない点は保たれますが、**出現頻度の偏りを打ち消すものが無くなります**。
どの疾患でも有名な遺伝子が上に来やすくなるので、8 章の「頻度のみ」対照で必ず確認してください。

In [ ]:
import subprocess

if BACKEND == "ollama" and OLLAMA_HOST is None:
    from backends import discover_ollama_host
    OLLAMA_HOST = discover_ollama_host()
    print("Ollama:", OLLAMA_HOST or "見つかりません（下の診断を参照）")
    OLLAMA_HOST = OLLAMA_HOST or "http://localhost:11434"

if not MODEL:
    print("MODEL が未設定のため飛ばしました。")
else:
    cmd = [sys.executable, os.path.join(SKILL, "scripts", "check_backend.py"),
           "--backend", BACKEND, "--model", MODEL]
    if BACKEND == "ollama":
        cmd += ["--host", OLLAMA_HOST]
    if BACKEND == "llamacpp":
        cmd += ["--no-load"]      # 重みの読み込みは 5 章で行う
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or r.stderr)

### Docker の Ollama を使う場合

**接続。** コンテナがポートを公開していれば（`-p 11434:11434`）、ホストからは
`http://localhost:11434` で届きます。上のセルは localhost →
`host.docker.internal`（Docker Desktop）→ `172.17.0.1`（Linux のブリッジ）の
順に自動で探します。このノートブック自体も別のコンテナで動いている場合、
`localhost` は**自分自身**を指すので、後ろ 2 つが必要になります。

**重みの置き場所。ここが Docker 特有の落とし穴です。** 公式の推奨どおり

```bash
docker run -d -p 11434:11434 -v ollama:/root/.ollama --name ollama ollama/ollama
```

と**名前付きボリューム**で起動していると、GGUF ファイルはホストの
ファイルシステム上に素直には現れません。Linux では
`/var/lib/docker/volumes/ollama/_data` 以下で通常 root 権限が要り、
Docker Desktop（Mac / Windows）では VM の中なのでホストからは**まったく見えません**。

つまり「Ollama の重みをそのまま llama.cpp で読む」が、この構成だと成立しません。
選択肢は 2 つです。

| | 方法 | 代償 |
|---|---|---|
| 1 | バインドマウントに変える<br>`-v ~/.ollama:/root/.ollama` で起動し直す | 再起動のみ。**ファイルは複製されない**ので基本はこちら |
| 2 | `docker cp` でコピーする | ディスクを二重に使う（27B の Q4 で約 17GB） |

上のセルの診断が、実際のコンテナ名と digest を使った `docker cp` の
コマンドをそのまま出します。

### Ollama の重みを llama.cpp で使う

`BACKEND = "llamacpp"` にして、`MODEL` は Ollama と同じ名前（`gemma3:27b` など）のままで
構いません。マニフェストから GGUF の場所を自動で引きます。

In [ ]:
if BACKEND in ("ollama", "llamacpp") and MODEL:
    from backends import (BackendError, docker_ollama_containers,
                          resolve_ollama_gguf)
    containers = docker_ollama_containers()
    if containers:
        print("Docker のコンテナ:", ", ".join(containers))
    try:
        blob = resolve_ollama_gguf(MODEL)
        gb = os.path.getsize(blob) / (1024 ** 3)
        print(f"GGUF  : {blob}")
        print(f"サイズ: {gb:.1f} GB")
        print()
        print('BACKEND = "llamacpp" に変えれば、このファイルで段階 1 も動きます。')
        print("（ファイルの複製は不要です）")
    except BackendError as e:
        print(e)
else:
    print("Ollama の重み探索は ollama / llamacpp のときだけ行います。")

## 4. トークナイザの確認（モデルが必要）

数秒で終わります。2 つの静かな失敗を防げます。

- 遺伝子記号がどう分割されるか（先頭が同じ記号ほど、生成時に取り違えやすい）
- ` A`〜` E` がそれぞれ 1 トークンか（段階 2 の前提）

In [ ]:
if not HAVE_MODEL:
    print("MODEL が未設定のため飛ばしました。")
elif BACKEND != "transformers":
    print(f"{BACKEND} では Hugging Face のトークナイザを直接見られないため飛ばしました。")
    print("段階 2 のラベルが読めているかは、3.5 章の check_backend.py が")
    print("実際の応答から確認しています（labels seen の行）。")
else:
    import subprocess
    cmd = [sys.executable, os.path.join(SKILL, "scripts", "check_tokenizer.py"),
           "--model", MODEL, "--genes"] + genes[:6]
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)

## 5. モデルを読み込む

`GeneRanker` は重みを 1 回だけ読み、段階 1 と段階 2 で使い回します。
疾患に依存しない「引く項」も 1 回だけ計算して再利用します。
遺伝子 1000 個 × 疾患 200 件なら、40 万回の計算が 20 万 1 千回になります。

In [ ]:
from rank import GeneRanker

ranker = None
if not HAVE_MODEL:
    print("MODEL が未設定のため飛ばしました。")
else:
    kw = {"host": OLLAMA_HOST} if BACKEND == "ollama" else {}
    ranker = GeneRanker(MODEL, backend=BACKEND, dtype="bfloat16",
                        batch_size=32, **kw)
    print(f"準備完了（backend={BACKEND}、重みの読み込みは最初の採点時）")
    if BACKEND == "ollama":
        print("段階 1 は使えないため、段階 2 を全候補に対して回します。")

## 6. ランキングを実行

段階 1 と段階 2 が続けて走ります。

In [ ]:
result = None
if ranker is None:
    print("モデル未設定のため飛ばしました。")
else:
    result = ranker.rank(
        disease=DISEASE,
        genes=genes,
        top_k=TOP_K,
        group_size=GROUP_SIZE,
        rotations=ROTATIONS,
        margin_threshold=MARGIN_THRESHOLD,
    )
    print("完了")

### 段階 1 の結果（PMI）

In [ ]:
def show_table(rows, cols):
    """pandas があれば表として、無ければ素のテキストで表示する。"""
    if not rows:
        print("（該当なし）")
        return
    if DEPS["pandas"]:
        import pandas as pd
        df = pd.DataFrame(rows, columns=cols)
        try:
            display(df)                      # ノートブック上
        except NameError:
            print(df.to_string(index=False))  # 素の python で実行した場合
        return
    w = [max(len(str(c)), *(len(str(r.get(c, ""))) for r in rows)) for c in cols]
    line = "  ".join(str(c).ljust(x) for c, x in zip(cols, w))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(x) for c, x in zip(cols, w)))

if result and result["stage1"]:
    print("PMI = 疾患ありの点数 − 疾患なしの点数（＝出現頻度の分を引いたもの）\n")
    show_table(result["stage1"][:10], ["gene", "pmi", "cond", "neutral"])
    print("\n見るべき点：cond（引く前）で上位の有名遺伝子が、pmi で下がっていれば補正が効いています。")
elif result:
    print("候補が少ないため段階 1 は実行されていません。")
else:
    print("モデル未設定のため飛ばしました。")

### 段階 2 の結果と最終判定

In [ ]:
if result:
    show_table([{"gene": r["gene"], "score": r["score"]}
                for r in result["stage2"][:10]], ["gene", "score"])
    print()
    if result["call"]:
        print(f"判定           : {result['call']}")
    else:
        print(f"判定           : なし（{result['abstain_reason']}）")
    print(f"1 位と 2 位の差 : {result['margin']}")
    print(f"順序への頑健さ  : {result['rank_stability']}  (1.0 に近いほど良い)")
    print()
    print("差が小さいときは間違えている確率が高いので、無理に答えを出さないほうが役に立ちます。")
    print("『3 割は判定しない』パイプラインのほうが、自信満々に間違えるものより有用です。")
    print("順序への頑健さが低い場合、モデルは遺伝子ではなく選択肢の位置を見ています。")
else:
    print("モデル未設定のため飛ばしました。")

## 7. 複数の疾患をまとめて処理

同じ遺伝子リストを使う場合、`GeneRanker` は 1 つを使い回してください。
「引く項」の再計算を避けられます。

In [ ]:
DISEASES = [
    "Huntington disease",
    "Cystic fibrosis",
    "Sickle cell disease",
    "Alzheimer disease",
]

OUT_PATH = "stage_results.jsonl"

results = []
if ranker is None:
    print("モデル未設定のため飛ばしました。")
else:
    results = ranker.rank_many(
        DISEASES, genes,
        top_k=TOP_K, group_size=GROUP_SIZE, rotations=ROTATIONS,
        margin_threshold=MARGIN_THRESHOLD,
    )
    with open(OUT_PATH, "w") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    show_table([{"disease": r["disease"], "call": r["call"] or "判定なし",
                 "margin": r["margin"], "stability": r["rank_stability"]}
                for r in results], ["disease", "call", "margin", "stability"])
    print(f"\n{OUT_PATH} に保存しました")

## 8. 評価と対照実験

**ここを飛ばすと、数字は何の意味も持ちません。**

正解データ（`gold.jsonl`）と比べます。同時に、比べるべき「下限」も必ず並べて出します。

| 対照 | 何がわかるか |
|---|---|
| ランダム | まぐれ当たりの水準。1000 候補から 50 件選ぶなら 5% |
| 疾患シャッフル | 疾患名を入れ替えても順位が変わらないなら、モデルは疾患を読んでいない |
| 頻度のみ | 論文での出現回数だけで並べたもの。これに勝てないなら PMI の補正が効いていない |
| データベースのみ | Open Targets の関連度スコアだけ。**これに勝てないなら LLM を外すべき** |

最後の 1 つはこのノートブックでは計算されません。別途取って比べてください。
Open Targets は大規模な遺伝子-疾患関連づけをすでに十分うまくやっており、
LLM の価値は「文章には書かれているが、整備済みデータには入っていない関連」に限られます。

In [ ]:
GOLD_PATH = os.path.join(EXAMPLES, "gold.jsonl")

if not results:
    print("予測結果がないため飛ばしました。")
elif not DEPS["numpy"]:
    print("numpy が必要です。")
else:
    import evaluate as ev

    preds = [json.loads(l) for l in open(OUT_PATH) if l.strip()]
    gold  = [json.loads(l) for l in open(GOLD_PATH) if l.strip()]
    ks = [1, 3, 5]

    m = ev.evaluate(preds, gold, ks)
    print(f"=== 本体（{m['n_diseases']} 疾患、平均 {m['mean_candidates']:.0f} 候補）===")
    ev.show("system", m, ks)
    print(f"\n  同族取り違え率 : {m['family_confusion_rate']:.3f}"
          f"  ({m['family_confusion_count']}/{m['top1_wrong_count']})")
    print("  これはこの構成が潰そうとしている失敗そのもの。ほぼ 0 になるはずです。")

    print("\n=== 対照 ===")
    rnd = ev.random_baseline(m["mean_candidates"], ks)
    print("  " + f"{'ランダム':<22} " +
          "  ".join(f"R{k}={rnd[f'@{k}']:.3f}" for k in ks))
    sh = ev.shuffle_control(preds, gold, ks)
    ev.show("疾患シャッフル", sh, ks)
    if sh["mrr"] > 0.5 * m["mrr"]:
        print("  → 入れ替えてもほとんど落ちていません。疾患を読めていない疑いがあります。")

### 敵対的な検証セット

本当の実力を測るには、**わざと似た記号を混ぜた**セットを作ります。
`GPR52` が正解なら、`GPR55` `GPR56` `GPR35` を選択肢に入れる。
無作為な選択肢を使うより、はるかによく実力が分かります。

正解率は大きく下がるはずです。下がらないなら、そのテストが簡単すぎた可能性を疑ってください。

また、疾患名と遺伝子名が似ているもの（Huntington と `HTT` など）は、
生物学的な知識ではなく**言葉の一致**で当たります。正解データに印を付けておき、
それを含む場合と除いた場合の両方を報告してください。

## 9. 実運用に入る前に

**エビデンスの裏取り。** LLM の点数は「仮説」であって「証拠」ではありません。
利用者に見せる前に、PubTator3 や Open Targets で裏付けを取ってください。
裏付けが 0 件のものは、黙って捨てるのではなく**印を付けて**出します。
LLM だけが拾った関連は、人間が確認する価値のある面白い候補だからです。

**規模。** ここで使った 4 疾患では何も結論できません。100 組以上の既知ペアで測ってください。

**しきい値の決め方。** `MARGIN_THRESHOLD` は検証用データで決めます。テストデータで
決めてはいけません。正解率と「判定しない率」のグラフを描き、動作点は利用者と一緒に選びます。
間違いのコストが「実験 1 本の無駄」なのか「一瞥の無駄」なのかで、適切な点は変わります。

**ライセンス。** モデルの重み・学習データ・データベースは、それぞれ別の条件です。
商用の予定があるなら、追加学習に投資する**前**に確認してください。
MeditronFO の学習データは研究用途で、モデルは臨床利用向けに承認されていません。
研究目的の遺伝子絞り込みと臨床判断支援は別の話ですが、最終的に臨床製品を目指すなら
早い段階で確認しておくほうが安上がりです。詳細は `references/data-sources.md`。
条件は変わるので、必ず一次情報で確認すること。